In [5]:
import cv2
import numpy as np
import os

def nothing(x):
    pass

# 1. Using the 'r' for a raw string, and pointing to the ORIGINAL dataset image
img_path = r'C:\Users\vivin\Downloads\egg.v1i.folder\train\egg\not_damaged_2_jpg.rf.a4629fbffb9457cd3dda5a1dfc6e5365.jpg'

if not os.path.exists(img_path):
    print(f"CRITICAL ERROR: Image not found at {img_path}")
else:
    image = cv2.imread(img_path)
    image = cv2.resize(image, (500, 500))
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    cv2.namedWindow('Trackbars')
    cv2.createTrackbar('Lower H', 'Trackbars', 0, 179, nothing)
    cv2.createTrackbar('Lower S', 'Trackbars', 0, 255, nothing)
    cv2.createTrackbar('Lower V', 'Trackbars', 150, 255, nothing)
    cv2.createTrackbar('Upper H', 'Trackbars', 179, 179, nothing)
    cv2.createTrackbar('Upper S', 'Trackbars', 50, 255, nothing) 
    cv2.createTrackbar('Upper V', 'Trackbars', 255, 255, nothing)

    while True:
        l_h = cv2.getTrackbarPos('Lower H', 'Trackbars')
        l_s = cv2.getTrackbarPos('Lower S', 'Trackbars')
        l_v = cv2.getTrackbarPos('Lower V', 'Trackbars')
        u_h = cv2.getTrackbarPos('Upper H', 'Trackbars')
        u_s = cv2.getTrackbarPos('Upper S', 'Trackbars')
        u_v = cv2.getTrackbarPos('Upper V', 'Trackbars')

        lower_bound = np.array([l_h, l_s, l_v])
        upper_bound = np.array([u_h, u_s, u_v])

        mask = cv2.inRange(hsv, lower_bound, upper_bound)
        
        cv2.imshow('Original', image)
        cv2.imshow('Mask (Tune this until egg is white, background is black)', mask)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            print(f"Final Lower Bound: {lower_bound}")
            print(f"Final Upper Bound: {upper_bound}")
            break

cv2.destroyAllWindows()

Final Lower Bound: [  0   0 116]
Final Upper Bound: [179  35 255]


In [ ]:
import cv2
import numpy as np

# 1. Load the image
img_path = r'C:\Users\vivin\Downloads\egg.v1i.folder\train\egg\not_damaged_2_jpg.rf.a4629fbffb9457cd3dda5a1dfc6e5365.jpg'
image = cv2.imread(img_path)
original_resized = cv2.resize(image, (500, 500))

# 2. Convert to HSV and apply your tuned mask
hsv = cv2.cvtColor(original_resized, cv2.COLOR_BGR2HSV)
lower_bound = np.array([0, 0, 116])
upper_bound = np.array([179, 35, 255])
raw_mask = cv2.inRange(hsv, lower_bound, upper_bound)

# 3. Find contours to isolate the egg from the noise
# RETR_EXTERNAL gets only the outer boundaries
contours, _ = cv2.findContours(raw_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 4. Create a solid black canvas for the final output
clean_mask = np.zeros_like(raw_mask)
 
if contours:
    # Identify the contour with the maximum pixel area (the egg)
    largest_contour = max(contours, key=cv2.contourArea)
    
    # Draw the largest contour filled completely (-1) in white (255)
    cv2.drawContours(clean_mask, [largest_contour], -1, 255, -1)

# 5. Display the before and after
cv2.imshow('Raw Mask (with noise)', raw_mask)
cv2.imshow('Cleaned Mask (Largest Contour Only)', clean_mask)
cv2.waitKey(0)
cv2.destroyAllWindows()

KeyboardInterrupt: 

: 

In [ ]:
import cv2
import numpy as np
import glob

# 1. Point to the entire training folder using a wildcard (*.jpg)
folder_path = r'C:\Users\vivin\Downloads\egg.v1i.folder\train\egg\*.jpg'
image_files = glob.glob(folder_path)

for img_path in image_files:
    image = cv2.imread(img_path)
    if image is None: 
        continue
    
    # 2. Process image
    resized = cv2.resize(image, (500, 500))
    hsv = cv2.cvtColor(resized, cv2.COLOR_BGR2HSV)
    
    # 3. Apply your tuned HSV threshold
    lower_bound = np.array([0, 0, 116])
    upper_bound = np.array([179, 35, 255])
    raw_mask = cv2.inRange(hsv, lower_bound, upper_bound)
    
    # 4. Filter by largest contour
    contours, _ = cv2.findContours(raw_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    clean_mask = np.zeros_like(raw_mask)
    
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        cv2.drawContours(clean_mask, [largest_contour], -1, 255, -1)
        
    # 5. Show original (converted to grayscale for easy comparison) alongside the mask
    gray_original = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    combined_view = np.hstack([gray_original, clean_mask])
    
    cv2.imshow('Left: Original | Right: Cleaned Mask (Press any key for next, Q to quit)', combined_view)
    
    # Wait for a key press. If 'q' is pressed, break the loop.
    if cv2.waitKey(0) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

In [ ]:
import cv2
import numpy as np
import glob

folder_path = r'C:\Users\vivin\Downloads\egg.v1i.folder\train\egg\*.jpg'
image_files = glob.glob(folder_path)

for img_path in image_files:
    image = cv2.imread(img_path)
    if image is None: 
        continue
    
    resized = cv2.resize(image, (500, 500))
    
    # 1. Convert to grayscale instead of HSV
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    
    # 2. Apply a heavy Gaussian Blur to smooth out the textures of the hand/background
    blurred = cv2.GaussianBlur(gray, (11, 11), 0)
    
    # 3. Apply Otsu's automatic thresholding
    # The '0' is a placeholder; Otsu calculates the real threshold value automatically
    _, raw_mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # 4. Filter by largest contour (same as before)
    contours, _ = cv2.findContours(raw_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    clean_mask = np.zeros_like(raw_mask)
    
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        cv2.drawContours(clean_mask, [largest_contour], -1, 255, -1)
        
    combined_view = np.hstack([gray, clean_mask])
    cv2.imshow('Left: Grayscale | Right: Otsu Cleaned Mask', combined_view)
    
    if cv2.waitKey(0) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

In [1]:
import cv2
import numpy as np
import glob

folder_path = r'C:\Users\vivin\Downloads\egg.v1i.folder\train\egg\*.jpg'
image_files = glob.glob(folder_path)

for img_path in image_files:
    image = cv2.imread(img_path)
    if image is None: 
        continue
    
    resized = cv2.resize(image, (500, 500))
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    
    # 1. Blur heavily to erase skin texture (fingerprints, creases)
    blurred = cv2.GaussianBlur(gray, (9, 9), 0)
    
    # 2. Canny Edge Detection
    # These numbers (lower & upper thresholds) dictate how "sharp" an edge must be to count
    edges = cv2.Canny(blurred, 30, 100)
    
    # 3. Morphological Dilation
    # Canny often leaves tiny gaps in the outline. Dilation thickens the lines to close the gaps.
    kernel = np.ones((5, 5), np.uint8)
    dilated_edges = cv2.dilate(edges, kernel, iterations=2)
    
    # 4. Find the largest contour (the closed loop) and fill it in
    contours, _ = cv2.findContours(dilated_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    clean_mask = np.zeros_like(gray)
    
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        cv2.drawContours(clean_mask, [largest_contour], -1, 255, -1)
        
    # Show the raw edges in the middle, and the final filled mask on the right
    combined_view = np.hstack([gray, edges, clean_mask])
    cv2.imshow('Gray | Raw Edges | Filled Mask (Press key for next, Q to quit)', combined_view)
    
    if cv2.waitKey(0) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

In [5]:
import cv2
import numpy as np
import glob
import os

# 1. Corrected paths (removed extra quotes, added *.jpg wildcard)
eval_folder = r'C:\Users\vivin\Downloads\egg.v1i.folder\valid\egg\*.jpg'
sam_mask_folder = r'C:\Users\vivin\Downloads\sam_masks_valid\content\runs\segment\predict'
output_bad_samples = r'C:\Users\vivin\Downloads\bad_samples'

os.makedirs(output_bad_samples, exist_ok=True)
image_files = glob.glob(eval_folder)

total_iou = 0
processed_count = 0
bad_threshold = 0.70 

for img_path in image_files:
    filename = os.path.basename(img_path)
    sam_path = os.path.join(sam_mask_folder, filename)
    
    if not os.path.exists(sam_path):
        continue

    # --- Classical Canny Pipeline ---
    image = cv2.imread(img_path)
    if image is None: continue
    
    resized = cv2.resize(image, (500, 500))
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (9, 9), 0)
    edges = cv2.Canny(blurred, 30, 100)
    kernel = np.ones((5, 5), np.uint8)
    dilated_edges = cv2.dilate(edges, kernel, iterations=2)
    contours, _ = cv2.findContours(dilated_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    my_mask = np.zeros_like(gray)
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        cv2.drawContours(my_mask, [largest_contour], -1, 255, -1)

    # --- Load SAM Ground Truth ---
    sam_img = cv2.imread(sam_path)
    if sam_img is None: continue
    
    sam_resized = cv2.resize(sam_img, (500, 500))
    sam_gray = cv2.cvtColor(sam_resized, cv2.COLOR_BGR2GRAY)
    _, sam_mask = cv2.threshold(sam_gray, 10, 255, cv2.THRESH_BINARY)

    # --- Calculate IoU ---
    intersection = cv2.bitwise_and(my_mask, sam_mask)
    union = cv2.bitwise_or(my_mask, sam_mask)
    
    intersection_area = np.count_nonzero(intersection)
    union_area = np.count_nonzero(union)
    
    iou = 0 if union_area == 0 else intersection_area / union_area
    total_iou += iou
    processed_count += 1

    # --- Save bad samples for the PDF report ---
    if iou < bad_threshold:
        combined_view = np.hstack([gray, my_mask, sam_mask])
        cv2.putText(combined_view, f"IoU: {iou:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        cv2.imwrite(os.path.join(output_bad_samples, f"fail_{filename}"), combined_view)

print(f"\n--- Evaluation Complete ---")
print(f"Images Processed: {processed_count}")
if processed_count > 0:
    print(f"Average IoU on Eval Split: {(total_iou / processed_count):.4f}")
print(f"Saved poorly segmented samples to: {output_bad_samples}")


--- Evaluation Complete ---
Images Processed: 22
Average IoU on Eval Split: 0.1353
Saved poorly segmented samples to: C:\Users\vivin\Downloads\bad_samples


In [7]:
import cv2
import numpy as np
import glob
import os

print("--- Starting Evaluation Script ---")

eval_folder = r'C:\Users\vivin\Downloads\egg.v1i.folder\valid\egg\*.jpg'
sam_mask_folder = r'C:\Users\vivin\Downloads\pure_masks\content\pure_masks'
output_bad_samples = r'C:\Users\vivin\Downloads\bad_samples'

os.makedirs(output_bad_samples, exist_ok=True)
image_files = glob.glob(eval_folder)

print(f"Found {len(image_files)} original images in the valid folder.")

total_iou = 0
processed_count = 0
bad_threshold = 0.70 

for img_path in image_files:
    filename = os.path.basename(img_path)
    sam_path = os.path.join(sam_mask_folder, filename)
    
    if not os.path.exists(sam_path):
        print(f"[SKIP] No pure mask found for: {filename}")
        continue

    # --- Classical Canny Pipeline ---
    image = cv2.imread(img_path)
    if image is None: continue
    
    resized = cv2.resize(image, (500, 500))
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (9, 9), 0)
    edges = cv2.Canny(blurred, 30, 100)
    kernel = np.ones((5, 5), np.uint8)
    dilated_edges = cv2.dilate(edges, kernel, iterations=2)
    contours, _ = cv2.findContours(dilated_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    my_mask = np.zeros_like(gray)
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        cv2.drawContours(my_mask, [largest_contour], -1, 255, -1)

    # --- Load Pure SAM Ground Truth ---
    sam_img = cv2.imread(sam_path, cv2.IMREAD_GRAYSCALE)
    if sam_img is None: continue
    
    sam_resized = cv2.resize(sam_img, (500, 500))
    _, sam_mask = cv2.threshold(sam_resized, 127, 255, cv2.THRESH_BINARY)

    # --- Calculate IoU ---
    intersection = cv2.bitwise_and(my_mask, sam_mask)
    union = cv2.bitwise_or(my_mask, sam_mask)
    
    intersection_area = np.count_nonzero(intersection)
    union_area = np.count_nonzero(union)
    
    iou = 0 if union_area == 0 else intersection_area / union_area
    total_iou += iou
    processed_count += 1
    
    print(f"[OK] Processed {filename} | Score: {iou:.2f}")

    # --- Save bad samples for the PDF report ---
    if iou < bad_threshold:
        combined_view = np.hstack([gray, my_mask, sam_mask])
        cv2.putText(combined_view, f"IoU: {iou:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        cv2.imwrite(os.path.join(output_bad_samples, f"fail_{filename}"), combined_view)

print(f"\n--- Evaluation Complete ---")
print(f"Images Processed: {processed_count}")
if processed_count > 0:
    print(f"Average IoU on Eval Split: {(total_iou / processed_count):.4f}")
print(f"Saved poorly segmented samples to: {output_bad_samples}")

--- Starting Evaluation Script ---
Found 22 original images in the valid folder.
[OK] Processed not_damaged_102_jpg.rf.8d754cbc174c6cb6fb1b5c1929edcbb1.jpg | Score: 0.08
[OK] Processed not_damaged_107_jpg.rf.8497e164672ad2319ef9ea157023304e.jpg | Score: 0.02
[OK] Processed not_damaged_109_jpg.rf.4d74eb94b1d03ebd3e7c442e58699304.jpg | Score: 0.07
[OK] Processed not_damaged_115_jpg.rf.5e04f25a50022be8a5f0dbcfe42bc26f.jpg | Score: 0.22
[OK] Processed not_damaged_119_jpg.rf.6758625c4b939166ca105d0509fc569e.jpg | Score: 0.43
[OK] Processed not_damaged_11_jpg.rf.95e1bc4944748cfd1a56fe23a42b6ae8.jpg | Score: 0.32
[OK] Processed not_damaged_120_jpg.rf.dfe91ad8ce891c84e495a8d59bbe01f5.jpg | Score: 0.04
[OK] Processed not_damaged_122_jpg.rf.b44d8e5a76cadeff9b2d8eaa85309d09.jpg | Score: 0.02
[OK] Processed not_damaged_126_jpg.rf.2c3342cf9ca7ab55f6682ef198e98bbd.jpg | Score: 0.01
[OK] Processed not_damaged_130_jpg.rf.7289d26d022f33aa74b3517f40d6a127.jpg | Score: 0.15
[OK] Processed not_damaged_133

In [1]:
import cv2
import glob
import os

# 1. Define paths
image_folder = r'C:\Users\vivin\Downloads\egg.v1i.folder\train\egg\*.jpg'
output_video_path = r'C:\Users\vivin\Downloads\egg_test_video.mp4'

# 2. Grab all image files
image_files = glob.glob(image_folder)

if not image_files:
    print("No images found! Check your path.")
else:
    print(f"Found {len(image_files)} images. Stitching into video...")
    
    # 3. Setup the Video Writer
    # We use 'mp4v' codec for standard mp4 files on Windows
    fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
    fps = 5.0 # 5 frames per second so you can actually see the tracking
    frame_size = (500, 500) # Must match the resize in our tracking script
    
    out = cv2.VideoWriter(output_video_path, fourcc, fps, frame_size)
    
    # 4. Loop through images and write them to the video
    for img_path in image_files:
        img = cv2.imread(img_path)
        if img is not None:
            # Resize to ensure every frame is exactly 500x500
            resized = cv2.resize(img, frame_size)
            out.write(resized)
            
    # 5. Save and close
    out.release()
    print(f"Success! Video saved to: {output_video_path}")

Found 75 images. Stitching into video...
Success! Video saved to: C:\Users\vivin\Downloads\egg_test_video.mp4


In [5]:
import cv2
import numpy as np
import os

# 1. Verify file exists before OpenCV touches it
video_path = r'C:\Users\vivin\Downloads\egg_test_video.mp4' 

if not os.path.exists(video_path):
    print(f"CRITICAL ERROR: No file exists at {video_path}")
    print("Check your Downloads folder to see where the video stitcher actually saved it.")
else:
    print(f"File found at {video_path}. Size: {os.path.getsize(video_path)} bytes.")
    print("Attempting to open video stream...")

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("ERROR: OpenCV found the file, but cannot read the video codec.")

# Proceed with tracking if opened successfully
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("End of video stream reached.")
        break

    frame = cv2.resize(frame, (500, 500))
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (9, 9), 0)
    edges = cv2.Canny(blurred, 30, 100)
    kernel = np.ones((5, 5), np.uint8)
    dilated_edges = cv2.dilate(edges, kernel, iterations=2)
    contours, _ = cv2.findContours(dilated_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        if cv2.contourArea(largest_contour) > 500:
            x, y, w, h = cv2.boundingRect(largest_contour)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            
            M = cv2.moments(largest_contour)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                cv2.circle(frame, (cx, cy), 5, (0, 0, 255), -1)
                cv2.putText(frame, f"Tracking: X:{cx} Y:{cy}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    cv2.imshow('Real-Time Egg Tracker', frame)

    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

File found at C:\Users\vivin\Downloads\egg_test_video.mp4. Size: 2051465 bytes.
Attempting to open video stream...
End of video stream reached.


In [1]:
import cv2
import numpy as np
import glob

# 1. Point directly to your validation images to simulate a video feed
video_frames = r'C:\Users\vivin\Downloads\egg.v1i.folder\valid\egg\*.jpg'
image_files = glob.glob(video_frames)

print(f"Simulating live video feed using {len(image_files)} frames...")
print("Press 'q' to quit.")

quit_video = False

# This outer loop makes the video repeat indefinitely
while not quit_video:
    
    # Loop through the 22 frames
    for img_path in image_files:
        frame = cv2.imread(img_path)
        if frame is None:
            continue

        frame = cv2.resize(frame, (500, 500))
        
        # --- Task 2 Canny Pipeline ---
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        blurred = cv2.GaussianBlur(gray, (9, 9), 0)
        edges = cv2.Canny(blurred, 30, 100)
        kernel = np.ones((5, 5), np.uint8)
        dilated_edges = cv2.dilate(edges, kernel, iterations=2)
        contours, _ = cv2.findContours(dilated_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # --- Task 4 Tracking Logic ---
        if contours:
            largest_contour = max(contours, key=cv2.contourArea)
            
            # Only track if the object is reasonably large
            if cv2.contourArea(largest_contour) > 500:
                # 1. Bounding Box (Draws a green rectangle)
                x, y, w, h = cv2.boundingRect(largest_contour)
                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
                
                # 2. Centroid / Center of Mass (Draws a red dot)
                M = cv2.moments(largest_contour)
                if M["m00"] != 0:
                    cx = int(M["m10"] / M["m00"])
                    cy = int(M["m01"] / M["m00"])
                    cv2.circle(frame, (cx, cy), 5, (0, 0, 255), -1)
                    
                    cv2.putText(frame, f"Tracking: X:{cx} Y:{cy}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        cv2.imshow('Real-Time Egg Tracker', frame)

        # 300ms delay per frame. If 'q' is pressed, set flag to exit completely.
        if cv2.waitKey(300) & 0xFF == ord('q'):
            quit_video = True
            break # Breaks the inner for-loop

cv2.destroyAllWindows()

Simulating live video feed using 22 frames...
Press 'q' to quit.
